# ARC 2026 P0 probe: Qwen3.8-27B-FP8 on 4xL4
Dev run only (not a submission). See artifacts/verification/arc-agi-2/new-approach-proposal-v1.md §6.


In [ ]:
%%writefile arc_llm.py
"""Pure helpers for the Qwen3.8 ARC probe (new-approach-proposal-v1 §6): prompt, answer parsing, dihedral + colour
augmentation and its inverse, and summary statistics. No I/O, no third-party imports, so it runs in the probe and in
local unit tests alike."""
import json
import math
import random
import re
import zlib

# --- grids -----------------------------------------------------------------------------------------------------------

def valid_grid(g):
    return (isinstance(g, list) and 0 < len(g) <= 30 and all(isinstance(r, list) for r in g)
            and 0 < len(g[0]) <= 30 and all(len(r) == len(g[0]) for r in g)
            and all(type(v) is int and 0 <= v <= 9 for r in g for v in r))


def transpose(g):
    return [list(r) for r in zip(*g)]


def rot90(g):  # clockwise
    return [list(r) for r in zip(*g[::-1])]


def dihedral(g, k):
    """k in 0..7: rotate k%4 quarter turns clockwise, then transpose if k >= 4."""
    for _ in range(k % 4):
        g = rot90(g)
    return transpose(g) if k >= 4 else [list(r) for r in g]


def dihedral_inverse(g, k):
    if k >= 4:
        g = transpose(g)
    for _ in range((4 - k % 4) % 4):
        g = rot90(g)
    return [list(r) for r in g]


def recolor(g, perm):
    return [[perm[v] for v in r] for r in g]


def augmentation(task_id, salt="p0-v1"):
    """Deterministic non-identity transform per task: dihedral k in 1..7 and a permutation of colours 1..9 (0 fixed,
    ARC background)."""
    rng = random.Random(zlib.crc32(f"{salt}:{task_id}".encode()))
    k = rng.randrange(1, 8)
    rest = list(range(1, 10))
    rng.shuffle(rest)
    perm = [0] + rest
    return {"k": k, "perm": perm}


def apply_aug(grid, aug):
    return recolor(dihedral(grid, aug["k"]), aug["perm"])


def invert_aug(grid, aug):
    inv = [0] * 10
    for a, b in enumerate(aug["perm"]):
        inv[b] = a
    return dihedral_inverse(recolor(grid, inv), aug["k"])


def augment_task(task, aug):
    return {"train": [{"input": apply_aug(p["input"], aug), "output": apply_aug(p["output"], aug)}
                      for p in task["train"]],
            "test": [{"input": apply_aug(p["input"], aug)} for p in task["test"]]}


# --- prompt / parse --------------------------------------------------------------------------------------------------

def grid_text(g):
    return "[" + ",\n ".join(json.dumps(r, separators=(",", ":")) for r in g) + "]"


def build_prompt(task, test_index):
    parts = ["You are solving an ARC-AGI puzzle. Each example maps an input grid to an output grid with the same "
             "hidden rule. Grids are lists of rows; each cell is a colour 0-9.", ""]
    for i, p in enumerate(task["train"], 1):
        parts += [f"Example {i} input:", grid_text(p["input"]), f"Example {i} output:", grid_text(p["output"]), ""]
    parts += ["Test input:", grid_text(task["test"][test_index]["input"]), "",
              "Find the rule, apply it to the test input, and give the test output grid. End your answer with the "
              "output grid as a JSON list of lists inside a ```json code block, and nothing after it."]
    return "\n".join(parts)


_FENCE = re.compile(r"```(?:json)?\s*(\[.*?\])\s*```", re.S)


def _candidates(text):
    for m in _FENCE.finditer(text):
        yield m.group(1)
    # bare outermost [[...]] blocks, last ones last
    depth, start = 0, None
    for i, ch in enumerate(text):
        if ch == "[":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "]" and depth:
            depth -= 1
            if depth == 0 and start is not None:
                yield text[start:i + 1]


def parse_answer(text):
    """Last valid grid found, scanning fenced blocks first and then every bare outermost list, so a valid bare list after
    the fence wins; None if there is none."""
    if not text:
        return None
    found = None
    for chunk in _candidates(text):
        try:
            g = json.loads(chunk)
        except (ValueError, RecursionError):
            continue
        if valid_grid(g):
            found = g
    return found


# --- statistics ------------------------------------------------------------------------------------------------------

def quantiles(xs, qs=(0.5, 0.9)):
    if not xs:
        return {f"p{int(q * 100)}": None for q in qs}
    s = sorted(xs)
    return {f"p{int(q * 100)}": s[min(len(s) - 1, max(0, math.ceil(q * len(s)) - 1))] for q in qs}


def km_quantiles(done, cens, qs=(0.5, 0.9)):
    """Kaplan-Meier quantiles of tokens-per-attempt with right-censoring (aborted/unfinished attempts are censored at
    the tokens they reached). None where the survival curve never drops that far."""
    events = sorted([(t, 1) for t in done] + [(t, 0) for t in cens])
    at_risk, surv, out, i = len(events), 1.0, {}, 0
    pending = list(qs)
    while i < len(events) and pending:
        t = events[i][0]
        d = c = 0
        while i < len(events) and events[i][0] == t:
            d += events[i][1]
            c += 1 - events[i][1]
            i += 1
        if d and at_risk:
            surv *= 1 - d / at_risk
        at_risk -= d + c
        while pending and 1 - surv >= pending[0]:
            out[f"km_p{int(pending.pop(0) * 100)}"] = t
    for q in pending:
        out[f"km_p{int(q * 100)}"] = None
    return out


def token_stats(rows):
    done = [r["completion_tokens"] for r in rows if r.get("state") == "done" and r.get("completion_tokens")]
    cens = [r["completion_tokens"] for r in rows if r.get("state") != "done" and r.get("completion_tokens")]
    out = {"n": len(rows), "n_done": len(done), "n_censored": len(cens),
           "mean_done": (sum(done) / len(done)) if done else None, "max_done": max(done) if done else None,
           "sum_all": sum(done) + sum(cens), "censored_over_40k": sum(1 for t in cens if t >= 40000),
           "censored_over_80k": sum(1 for t in cens if t >= 80000)}
    out.update(quantiles(done))
    out.update(km_quantiles(done, cens))
    return out


def arm_summary(rows):
    """rows: one effort arm. Accuracy is per test output; truncation = finish_reason 'length' among finished."""
    finished = [r for r in rows if r.get("state") == "done"]
    return {"tokens": token_stats(rows),
            "truncated": sum(1 for r in finished if r.get("finish_reason") == "length"),
            "parsed": sum(1 for r in finished if r.get("answer") is not None),
            "correct": sum(1 for r in finished if r.get("correct")),
            "finished": len(finished), "requests": len(rows)}


def pair_agreement(rows):
    """For an arm sampled twice per output: agreement count and its precision (agreeing answer correct)."""
    by = {}
    for r in rows:
        if r.get("state") == "done":
            by.setdefault(r["key"], []).append(r)
    agree = agree_ok = both = 0
    for rs in by.values():
        if len(rs) < 2:
            continue
        both += 1
        a, b = rs[0].get("answer"), rs[1].get("answer")
        if a is not None and a == b:
            agree += 1
            agree_ok += bool(rs[0].get("correct"))
    return {"pairs": both, "agree": agree, "agree_correct": agree_ok,
            "precision": (agree_ok / agree) if agree else None}


In [ ]:
%%writefile p0_probe.py
"""P0 probe for the Qwen3.8-27B-FP8 line (artifacts/verification/arc-agi-2/new-approach-proposal-v1.md §6).

One Kaggle 4xL4 commit run, offline. Measures, in order, with a hard self-imposed wall clock:
  1. environment, wheel hashes (SHA256SUMS + PyPI pins), model weight sha256 vs the HF LFS list;
  2. vLLM server start (TP=4, FP8 KV, TRITON_ATTN pinned, MTP; fallback ladder) and KV capacity from its log;
  3. long-reasoning throughput under load via /metrics, and per-request tokens, truncation, parse rate and accuracy
     for reasoning_effort xhigh x1 and low x2 on colour/dihedral-augmented ARC-AGI-2 public-eval tasks (keys30);
  4. if time remains, KV capacity and short-context throughput of a TP=2 replica.
Everything is written incrementally to $P0_OUT (summary.json, requests.jsonl, metrics.jsonl, probe.log, server logs).
Never touches the network (pip --no-index, HF offline). Exits 0 on every outcome so the outputs are downloadable."""
import glob
import hashlib
import json
import os
import re
import signal
import subprocess
import sys
import threading
import time
import zlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import arc_llm as L  # noqa: E402

T0 = time.time()
WALL = float(os.environ.get("P0_WALL_SECONDS", 3 * 3600 + 10 * 60))
OUT = Path(os.environ.get("P0_OUT", "/kaggle/working/p0"))
INPUT = Path(os.environ.get("P0_INPUT", "/kaggle/input"))
HERE = Path(__file__).resolve().parent
N_TASKS = int(os.environ.get("P0_TASKS", 20))
CAPS = {"xhigh": 120000, "low": 80000}
MAX_LEN = 131072
MAIN_STOP = T0 + WALL - 35 * 60   # no new main-phase requests after this
MAIN_ABORT = T0 + WALL - 28 * 60  # in-flight main-phase requests are cut here (recorded as censored)
TAIL_NEED = 20 * 60               # TP=2 tail only if this much remains before the hard end
END_RESERVE = 6 * 60              # always keep this for writing outputs
PYPI = {  # sha256 of the official PyPI wheels (pypi.org JSON API, 2026-10-12)
    "vllm-0.19.0-cp38-abi3-manylinux_2_31_x86_64.whl":
        "2d0e5fae45367bdbf111fcad68f4c0f8fdddd2f2fb643e52f0f2daebef7b41cf",
    "torch-2.10.0-3-cp312-cp312-manylinux_2_28_x86_64.whl":
        "98c01b8bb5e3240426dcde1446eed6f40c778091c8544767ef1168fc663a05a6",
}
KEYS30 = json.loads((HERE / "keys30.json").read_text())  # E1 keys30 (gcp-e1/make_inputs.py), public eval

HASH_POOL = []
ACTIVE = {}  # id(row) -> streaming response, closed by the watchdog at abort time
S = {"version": "p0-v1", "t0": T0, "wall_seconds": WALL, "phases": {}, "fatal": None}
_lock = threading.Lock()
ABORT = threading.Event()


def el():
    return round(time.time() - T0, 1)


def log(msg):
    line = f"[{el():8.1f}s] {msg}"
    print(line, flush=True)
    with _lock, (OUT / "probe.log").open("a") as f:
        f.write(line + "\n")


def save():
    with _lock:
        tmp = OUT / "summary.json.tmp"
        tmp.write_text(json.dumps(S, indent=1, default=str))
        tmp.replace(OUT / "summary.json")


def append(name, row):
    with _lock, (OUT / name).open("a") as f:
        f.write(json.dumps(row, default=str) + "\n")


def run(cmd, timeout=None, **kw):
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, **kw)
    return p.returncode, (p.stdout or "") + (p.stderr or "")


def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()


def find_dir(pattern):
    hits = sorted(glob.glob(str(INPUT / "**" / pattern), recursive=True))
    return Path(hits[0]).parent if hits else None


# --- 1. environment and integrity ------------------------------------------------------------------------------------

def phase_env():
    ph = S["phases"]["env"] = {"python": sys.version}
    for name, cmd in [("nvidia_smi", ["nvidia-smi"]), ("df", ["df", "-h", "/kaggle/working", "/tmp"]),
                      ("nproc", ["nproc"]), ("free", ["free", "-g"])]:
        try:
            ph[name] = run(cmd, timeout=30)[1][-3000:]
        except Exception as e:  # noqa: BLE001
            ph[name] = repr(e)
    comp, model, wheels = (find_dir("arc-agi_evaluation_challenges.json"), find_dir("layers-0.safetensors"),
                           find_dir("vllm-0.19.0-*.whl"))
    ph.update(comp_dir=str(comp), model_dir=str(model), wheel_dir=str(wheels))
    log(f"python {sys.version.split()[0]} comp={comp} model={model} wheels={wheels}")
    save()
    if not (comp and model and wheels):
        raise SystemExit("input mount missing")
    return comp, model, wheels


def verify_wheels(wheels):
    ph = S["phases"]["wheels"] = {}
    sums = {}
    for line in (wheels / "SHA256SUMS").read_text().splitlines():
        parts = line.split()
        if len(parts) == 2:
            sums[parts[1].lstrip("*")] = parts[0]
    files = sorted(p.name for p in wheels.glob("*.whl"))
    bad = [n for n in files if sums.get(n) != sha_file(wheels / n)]
    pypi_ok = {n: (sums.get(n) == h and (wheels / n).exists()) for n, h in PYPI.items()}
    ph.update(n_wheels=len(files), mismatched=bad, unlisted=[n for n in files if n not in sums], pypi_pins=pypi_ok)
    log(f"wheels {len(files)} mismatched={len(bad)} pypi_pins={pypi_ok}")
    save()
    if bad or not all(pypi_ok.values()):
        raise SystemExit("wheel hash check failed")


def start_model_hash(model):
    ref = json.loads((HERE / "qwen38_fp8_sha256.json").read_text())["files"]
    pool = ThreadPoolExecutor(max_workers=8)
    futs = {n: pool.submit(sha_file, model / n) for n in ref if (model / n).exists()}
    return ref, futs, pool


def finish_model_hash(model, ref, futs, pool):
    ph = S["phases"]["model_hash"] = {}
    missing = [n for n in ref if n not in futs]
    bad = [n for n, f in futs.items() if f.result() != ref[n][1]]
    pool.shutdown()
    sizes_bad = [n for n in futs if (model / n).stat().st_size != ref[n][0]]
    ph.update(files=len(ref), missing=missing, mismatched=bad, size_mismatched=sizes_bad)
    log(f"model sha256: {len(ref)} files, missing={missing} mismatched={bad}")
    save()
    if missing or bad or sizes_bad:
        raise SystemExit("model hash check failed")


def install(wheels):
    ph = S["phases"]["install"] = {}
    t = time.time()
    rc, out = run([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-index", "--find-links",
                   str(wheels), "--requirement", str(wheels / "requirements.lock")], timeout=1800)
    ph.update(rc=rc, seconds=round(time.time() - t), tail=out[-2500:])
    log(f"pip install rc={rc} in {ph['seconds']}s")
    save()
    if rc != 0:
        raise SystemExit("pip install failed")
    rc, out = run([sys.executable, "-c", "import json, torch, vllm, transformers; print(json.dumps({'torch': "
                   "torch.__version__, 'cuda': torch.version.cuda, 'cuda_ok': torch.cuda.is_available(), "
                   "'gpus': torch.cuda.device_count(), 'vllm': vllm.__version__, 'transformers': "
                   "transformers.__version__}))"], timeout=600)
    ph["import"] = out.strip().splitlines()[-1] if out.strip() else out
    log(f"import check rc={rc}: {ph['import']}")
    rc2, _ = run([sys.executable, "-c", "import flash_attn"], timeout=120)
    ph["flash_attn"] = rc2 == 0
    if rc2 != 0:  # vLLM forwards layer_idx to an attention impl that rejects it when flash-attn is absent (public
        # Apache-2.0 notebook delongyang/arc-agi-2-inference-qwen3-8-27b-fp8-within-4-hours documents this fix)
        rc3, root = run([sys.executable, "-c", "import os, vllm; print(os.path.dirname(vllm.__file__))"], timeout=300)
        root = root.strip().splitlines()[-1] if rc3 == 0 and root.strip() else ""
        patched, reverted = [], []
        for f in sorted(glob.glob(root + "/**/attention/*.py", recursive=True)) if root else []:
            text = Path(f).read_text()
            if "self.impl = impl_cls(" not in text or "extra_impl_args.pop('layer_idx', None)" in text:
                continue
            new = re.sub(r"^(\s*)self\.impl = impl_cls\(", r"\1extra_impl_args.pop('layer_idx', None)\n\1self.impl = impl_cls(",
                         text, count=1, flags=re.M)
            Path(f + ".bak").write_text(text)
            Path(f).write_text(new)
            if run([sys.executable, "-m", "py_compile", f], timeout=120)[0] != 0 or "extra_impl_args" not in text:
                Path(f).write_text(text)  # revert: does not compile, or the variable does not exist in this version
                reverted.append(f)
            else:
                patched.append(f)
        ph.update(attention_patch=patched, attention_patch_reverted=reverted, vllm_root=root)
        log(f"attention patch: patched={patched} reverted={reverted}")
    save()
    if rc != 0 or '"cuda_ok": true' not in ph["import"]:
        raise SystemExit("torch/vllm import or CUDA check failed")


# --- 2. vLLM server ------------------------------------------------------------------------------------------------------

def server_env(gpus=None, force_backend=True):
    env = dict(os.environ, PYTORCH_ALLOC_CONF="expandable_segments:True",
               VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS="1", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
               VLLM_NO_USAGE_STATS="1", DO_NOT_TRACK="1", VLLM_LOGGING_LEVEL="INFO")
    libcuda = "/usr/local/nvidia/lib64/libcuda.so"
    if os.path.exists(libcuda):  # same libcuda pin as the public notebook above
        env["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + env.get("LD_LIBRARY_PATH", "")
        env["LD_PRELOAD"] = libcuda
    if force_backend:  # pins the MTP drafter too (public notebook above, vLLM PR #53450)
        env["VLLM_FORCE_ATTN_BACKEND"] = "TRITON_ATTN"
    if gpus:
        env["CUDA_VISIBLE_DEVICES"] = gpus
    return env


SUPPORTED = None


def supported_flags():
    global SUPPORTED
    if SUPPORTED is None:
        rc, out = run([sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--help=all"], timeout=600)
        SUPPORTED = set(re.findall(r"(--[a-z0-9][a-z0-9-]+)", out)) if rc == 0 else set()
        S["phases"]["server_flags"] = {"rc": rc, "n": len(SUPPORTED)}
    return SUPPORTED


def start_server(model, port, tp, mtp, mm_off, gpus=None, tag="main", safe=False):
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", str(model),
           "--served-model-name", "q38", "--host", "127.0.0.1", "--port", str(port),
           "--tensor-parallel-size", str(tp), "--gpu-memory-utilization", "0.88" if safe else "0.93",
           "--max-model-len", str(MAX_LEN), "--max-num-seqs", "48", "--reasoning-parser", "qwen3",
           "--trust-remote-code"]
    if safe:  # last rung: default KV dtype and backend choice, eager mode, lower memory target
        cmd += ["--enforce-eager"]
    else:
        cmd += ["--kv-cache-dtype", "fp8", "--attention-backend", "TRITON_ATTN", "--no-async-scheduling"]
    if mm_off:
        cmd += ["--limit-mm-per-prompt", json.dumps({"image": 0, "video": 0})]
    if mtp:
        cmd += ["--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": mtp})]
    known = supported_flags()
    if known:  # drop flags this vLLM build does not list (logged), rather than fail every rung on one bad flag
        dropped, kept, i = [], [], 0
        while i < len(cmd):
            a = cmd[i]
            takes = i + 1 < len(cmd) and not cmd[i + 1].startswith("--")
            if a.startswith("--") and a not in known:
                dropped.append(a)
                i += 2 if takes and a not in ("--trust-remote-code", "--enforce-eager", "--no-async-scheduling") else 1
                continue
            kept.append(a)
            i += 1
        cmd = kept
        if dropped:
            log(f"dropped unsupported flags: {dropped}")
    path = OUT / f"server-{tag}-tp{tp}-mtp{mtp}-mm{int(mm_off)}{'-safe' if safe else ''}.log"
    fh = path.open("w")
    proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, text=True,
                            start_new_session=True, env=server_env(gpus, force_backend=not safe))
    return proc, path, cmd


def http(method, url, body=None, timeout=30):
    import requests
    r = requests.request(method, url, json=body, timeout=timeout)
    return r.status_code, r.text


def wait_ready(proc, port, deadline):
    while time.time() < deadline:
        if proc.poll() is not None:
            return False
        try:
            if http("GET", f"http://127.0.0.1:{port}/v1/models", timeout=5)[0] == 200:
                return True
        except Exception:  # noqa: BLE001
            pass
        time.sleep(5)
    return False


def smoke(port):
    body = {"model": "q38", "messages": [{"role": "user", "content": "Reply with the single word OK."}],
            "max_tokens": 32, "temperature": 0, "chat_template_kwargs": {"enable_thinking": False}}
    code, text = http("POST", f"http://127.0.0.1:{port}/v1/chat/completions", body, timeout=300)
    try:
        msg = json.loads(text)["choices"][0]["message"]
    except Exception:  # noqa: BLE001
        return False, text[-600:]
    txt = (msg.get("content") or "") + (msg.get("reasoning_content") or msg.get("reasoning") or "")
    return code == 200 and len(txt.strip()) > 0, text[-600:]


def kill(proc):
    if proc is None or proc.poll() is not None:
        return
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        proc.wait(timeout=60)
    except Exception:  # noqa: BLE001
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception:  # noqa: BLE001
            pass
    for _ in range(30):  # wait for VRAM release
        rc, out = run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], timeout=30)
        if rc == 0 and all(int(x) < 1500 for x in out.split() if x.strip().isdigit()):
            return
        time.sleep(4)


def kv_capacity(path):
    text = path.read_text(errors="ignore") if path.exists() else ""
    tok = re.findall(r"GPU KV cache size:\s*([\d,]+)\s*tokens", text)
    conc = re.findall(r"Maximum concurrency for\s*([\d,]+)\s*tokens per request:\s*([\d.]+)x", text)
    backend = sorted(set(re.findall(r"[Uu]sing (?:AttentionBackendEnum\.)?([A-Z][A-Z_]+)(?: attention)? backend", text)))
    return {"kv_tokens": int(tok[-1].replace(",", "")) if tok else None,
            "max_concurrency_at_len": float(conc[-1][1]) if conc else None, "backends_logged": backend}


def phase_server(model):
    ph = S["phases"]["server"] = {"attempts": []}
    for mtp, mm_off, safe in [(2, True, False), (0, True, False), (0, False, False), (0, True, True)]:
        if time.time() > T0 + WALL * 0.45:
            break
        proc, path, cmd = start_server(model, 8000, 4, mtp, mm_off, safe=safe)
        t = time.time()
        ready = wait_ready(proc, 8000, min(t + 30 * 60, T0 + WALL * 0.5))
        att = {"mtp": mtp, "mm_off": mm_off, "safe": safe, "ready": ready, "startup_seconds": round(time.time() - t),
               "log": path.name, "cmd": cmd}
        if ready:
            ok, resp = smoke(8000)
            att.update(smoke_ok=ok, smoke_resp=resp, **kv_capacity(path))
            if ok:
                ph["attempts"].append(att)
                ph["chosen"] = att
                log(f"server ready: mtp={mtp} mm_off={mm_off} kv={att['kv_tokens']} "
                    f"conc@131k={att['max_concurrency_at_len']} backends={att['backends_logged']}")
                save()
                return proc, att
        att["log_tail"] = path.read_text(errors="ignore")[-3000:] if path.exists() else ""
        ph["attempts"].append(att)
        log(f"server attempt failed: mtp={mtp} mm_off={mm_off} ready={ready}")
        save()
        kill(proc)
    raise SystemExit("no server configuration started")


# --- 3. metrics + requests -------------------------------------------------------------------------------------------

METRICS = ("vllm:generation_tokens_total", "vllm:prompt_tokens_total", "vllm:num_requests_running",
           "vllm:num_requests_waiting", "vllm:kv_cache_usage_perc", "vllm:gpu_cache_usage_perc",
           "vllm:num_preemptions_total", "vllm:spec_decode_num_accepted_tokens_total",
           "vllm:spec_decode_num_draft_tokens_total", "vllm:spec_decode_num_drafts_total")


def scrape(port):
    vals = {}
    try:
        code, text = http("GET", f"http://127.0.0.1:{port}/metrics", timeout=10)
    except Exception:  # noqa: BLE001
        return None
    if code != 200:
        return None
    for line in text.splitlines():
        if line.startswith("#"):
            continue
        m = re.match(r"([a-zA-Z_:]+)(\{[^}]*\})?\s+([0-9.eE+-]+)$", line)
        if m and m.group(1) in METRICS:
            vals[m.group(1)] = vals.get(m.group(1), 0.0) + float(m.group(3))
    return vals


def poller(port, stop, tag):
    while not stop.is_set():
        v = scrape(port)
        if v is not None:
            append("metrics.jsonl", {"t": el(), "tag": tag, **v})
        stop.wait(15)


def prompt_tokens(port, messages, effort):
    """Token count of the templated prompt; retries without template kwargs, then falls back to the character count
    (an upper bound for these digit/comma grids) so max_tokens never overflows the context."""
    for body in ({"model": "q38", "messages": messages,
                  "chat_template_kwargs": {"reasoning_effort": effort, "enable_thinking": True}},
                 {"model": "q38", "messages": messages}):
        try:
            code, text = http("POST", f"http://127.0.0.1:{port}/tokenize", body, timeout=30)
            if code == 200 and json.loads(text).get("count"):
                return json.loads(text)["count"], "tokenize"
        except Exception:  # noqa: BLE001
            pass
    return sum(len(m["content"]) for m in messages) + 300, "chars"


def one_request(job, port, abort_at, stop_at=None):
    import requests
    row = {k: job[k] for k in ("key", "task", "test_index", "effort", "rep", "aug_id", "phase")}
    row.update(state="not_started", t_queue=el())
    if time.time() > (MAIN_STOP if stop_at is None else stop_at) or ABORT.is_set():
        append("requests.jsonl", row)
        return row
    messages = [{"role": "user", "content": job["prompt"]}]
    n_prompt, how = prompt_tokens(port, messages, job["effort"])
    cap = min(CAPS[job["effort"]], MAX_LEN - n_prompt - 64)
    if cap < 4096:
        row.update(state="prompt_too_long", prompt_tokens_est=n_prompt)
        append("requests.jsonl", row)
        return row
    body = {"model": "q38", "messages": messages, "max_tokens": cap, "temperature": 1.0, "top_p": 0.95,
            "top_k": 20, "min_p": 0.0, "presence_penalty": 0.0, "seed": job["seed"], "stream": True,
            "stream_options": {"include_usage": True, "continuous_usage_stats": True},
            "chat_template_kwargs": {"reasoning_effort": job["effort"], "enable_thinking": True}}
    row.update(state="running", t_start=el(), max_tokens=cap, prompt_tokens_est=n_prompt, prompt_count_method=how)
    content, reasoning_chars, head, tail = [], 0, "", ""
    usage, finish = {}, None
    try:
        with requests.post(f"http://127.0.0.1:{port}/v1/chat/completions", json=body, stream=True,
                           timeout=(30, 600)) as r:
            ACTIVE[id(row)] = r
            if r.status_code != 200:
                row.update(state="http_error", error=r.text[-800:])
                return row
            for raw in r.iter_lines(decode_unicode=True):
                if time.time() > abort_at or ABORT.is_set():
                    row["state"] = "aborted"
                    break
                if not raw or not raw.startswith("data: "):
                    continue
                data = raw[6:]
                if data.strip() == "[DONE]":
                    break
                ev = json.loads(data)
                if ev.get("usage"):
                    usage = ev["usage"]
                for ch in ev.get("choices") or []:
                    d = ch.get("delta") or {}
                    rc = d.get("reasoning_content") or d.get("reasoning") or ""
                    if rc:
                        if len(head) < 1500:
                            head += rc[:1500 - len(head)]
                        tail = (tail + rc)[-1500:]
                        reasoning_chars += len(rc)
                    if d.get("content"):
                        content.append(d["content"])
                        if "t_first_content" not in row:
                            row["t_first_content"] = el()
                    if ch.get("finish_reason"):
                        finish = ch["finish_reason"]
    except Exception as e:  # noqa: BLE001
        stopped = time.time() > abort_at or ABORT.is_set()
        row.update(state="aborted" if stopped else "exception", error=repr(e)[-800:])
    finally:
        ACTIVE.pop(id(row), None)
    text = "".join(content)
    row.update(t_end=el(), prompt_tokens=usage.get("prompt_tokens"), completion_tokens=usage.get("completion_tokens"),
               finish_reason=finish, reasoning_chars=reasoning_chars, reasoning_head=head, reasoning_tail=tail,
               content_tail=text[-2000:])
    if row["state"] == "running":
        row["state"] = "done" if finish else "incomplete"
    ans = L.parse_answer(text) if row["state"] == "done" else None
    if ans is not None:
        ans = L.invert_aug(ans, job["aug"])
    row["answer"] = ans
    row["correct"] = ans is not None and ans == job["solution"]
    append("requests.jsonl", row)
    return row


def build_jobs(comp, phase="main"):
    """Per test output: xhigh and low rep0 on augmentation A, low rep1 on augmentation B (agreement across different
    augmentations is the stronger confidence signal); answers are inverted to the original frame before scoring."""
    ch = json.loads((comp / "arc-agi_evaluation_challenges.json").read_text())
    so = json.loads((comp / "arc-agi_evaluation_solutions.json").read_text())
    jobs, keys = [], [k for k in KEYS30 if k in ch][:N_TASKS]
    for k in keys:
        augs = {"A": L.augmentation(k), "B": L.augmentation(k, salt="p0-v1-B")}
        tasks = {a: L.augment_task(ch[k], g) for a, g in augs.items()}
        for i in range(len(ch[k]["test"])):
            for effort, rep, a in (("xhigh", 0, "A"), ("low", 0, "A"), ("low", 1, "B")):
                jobs.append({"key": f"{k}:{i}", "task": k, "test_index": i, "effort": effort, "rep": rep,
                             "aug_id": a, "phase": phase,
                             "seed": zlib.crc32(f"{k}:{i}:{effort}:{rep}".encode()) & 0x7FFFFFFF,
                             "prompt": L.build_prompt(tasks[a], i), "aug": augs[a], "solution": so[k][i]})
    return keys, jobs


def effort_check(port, jobs):
    """Does reasoning_effort reach the template? Token counts of the same prompt at xhigh vs low must differ."""
    msgs = [{"role": "user", "content": jobs[0]["prompt"]}]
    counts = {e: prompt_tokens(port, msgs, e) for e in ("xhigh", "low")}
    return {"counts": counts, "differ": counts["xhigh"] != counts["low"]}


def watchdog(proc, stop, abort_at):
    """Close live streams at abort time (a stalled stream otherwise blocks its worker up to the read timeout) and
    record a server death, which ends the phase early instead of failing every queued job slowly."""
    while not stop.is_set():
        if time.time() > abort_at:
            ABORT.set()
        if proc is not None and proc.poll() is not None:
            S["phases"].setdefault("main", {})["server_died_at"] = el()
            log("server process died")
            ABORT.set()
        if ABORT.is_set():
            for r in list(ACTIVE.values()):
                try:
                    r.close()
                except Exception:  # noqa: BLE001
                    pass
        stop.wait(5)


def phase_main(comp, att, port=8000, proc=None):
    ph = S["phases"]["main"] = {}
    keys, jobs = build_jobs(comp)
    y = att.get("max_concurrency_at_len") or 6.0
    conc = int(max(4, min(48, round(y * 1.8))))
    ph.update(tasks=keys, outputs=len({j["key"] for j in jobs}), jobs=len(jobs), concurrency=conc,
              t_start=el(), stop_at=round(MAIN_STOP - T0), abort_at=round(MAIN_ABORT - T0))
    ph["effort_check"] = effort_check(port, jobs)
    log(f"main: {len(keys)} tasks, {ph['outputs']} outputs, {len(jobs)} jobs, concurrency {conc}, "
        f"effort_check {ph['effort_check']}")
    save()
    stop = threading.Event()
    th = threading.Thread(target=poller, args=(port, stop, "main"), daemon=True)
    wd = threading.Thread(target=watchdog, args=(proc, stop, MAIN_ABORT), daemon=True)
    th.start()
    wd.start()
    rows = []
    with ThreadPoolExecutor(max_workers=conc) as pool:
        futs = [pool.submit(one_request, j, port, MAIN_ABORT) for j in jobs]
        for f in as_completed(futs):
            rows.append(f.result())
            if len(rows) % 10 == 0:
                ph["progress"] = {"rows": len(rows), "t": el()}
                save()
    stop.set()
    th.join(timeout=30)
    wd.join(timeout=30)
    ABORT.clear()
    ph["t_end"] = el()
    ph["arms"] = {e: L.arm_summary([r for r in rows if r["effort"] == e]) for e in ("xhigh", "low")}
    ph["low_pair_agreement"] = L.pair_agreement([r for r in rows if r["effort"] == "low"])
    by_key = {}
    for r in rows:
        by_key.setdefault(r["key"], []).append(r)
    ph["union_any_correct_outputs"] = sum(any(x.get("correct") for x in v) for v in by_key.values())
    ph["throughput"] = throughput("main", ph["t_start"], ph["t_end"], conc)
    log(f"main done: arms={json.dumps(ph['arms'])} throughput={json.dumps(ph['throughput'])}")
    save()


def throughput(tag, t_start, t_end, conc=None):
    p = OUT / "metrics.jsonl"
    rows = [json.loads(x) for x in p.read_text().splitlines() if x.strip()] if p.exists() else []
    rows = [r for r in rows if r.get("tag") == tag and "vllm:generation_tokens_total" in r]
    if len(rows) < 2:
        return {"samples": len(rows)}
    g = "vllm:generation_tokens_total"
    pairs = [(a, b) for a, b in zip(rows, rows[1:]) if b["t"] > a["t"]]
    rate = lambda ps: (sum(b[g] - a[g] for a, b in ps) / sum(b["t"] - a["t"] for a, b in ps)) if ps else None
    loaded = [(a, b) for a, b in pairs if conc and a.get("vllm:num_requests_running", 0) >= 0.8 * conc
              and a["t"] >= t_start + 300]
    per_req = [(b[g] - a[g]) / (b["t"] - a["t"]) / a["vllm:num_requests_running"] for a, b in loaded
               if a.get("vllm:num_requests_running")]
    a, b = rows[0], rows[-1]
    out = {"samples": len(rows), "overall_tok_per_s": round(rate(pairs), 1),
           "loaded_tok_per_s": round(rate(loaded), 1) if loaded else None, "loaded_seconds": round(sum(
               y["t"] - x["t"] for x, y in loaded)),
           "per_running_request_tok_per_s": round(sum(per_req) / len(per_req), 2) if per_req else None,
           "running_mean": round(sum(r.get("vllm:num_requests_running", 0) for r in rows) / len(rows), 2),
           "kv_usage_max": max(max(r.get("vllm:kv_cache_usage_perc", 0), r.get("vllm:gpu_cache_usage_perc", 0))
                               for r in rows),
           "preemptions": b.get("vllm:num_preemptions_total", 0) - a.get("vllm:num_preemptions_total", 0)}
    acc = b.get("vllm:spec_decode_num_accepted_tokens_total", 0) - a.get("vllm:spec_decode_num_accepted_tokens_total", 0)
    draft = b.get("vllm:spec_decode_num_draft_tokens_total", 0) - a.get("vllm:spec_decode_num_draft_tokens_total", 0)
    out["spec_acceptance"] = round(acc / draft, 3) if draft else None
    return out


# --- 4. TP=2 tail ------------------------------------------------------------------------------------------------------

def phase_tp2(model, comp, att):
    ph = S["phases"]["tp2"] = {}
    proc, path, _ = start_server(model, 8001, 2, att["mtp"], att["mm_off"], gpus="0,1", tag="tail")
    deadline = T0 + WALL - END_RESERVE - 5 * 60
    ready = wait_ready(proc, 8001, deadline)
    ph.update(ready=ready, **kv_capacity(path))
    if ready and time.time() < deadline:
        _, jobs = build_jobs(comp, phase="tail")
        xs = [j for j in jobs if j["effort"] == "xhigh"]
        conc = int(max(2, min(24, round((ph.get("max_concurrency_at_len") or 3) * 1.8))))
        stop = threading.Event()
        end = min(time.time() + 240, deadline)
        th = threading.Thread(target=poller, args=(8001, stop, "tp2"), daemon=True)
        wd = threading.Thread(target=watchdog, args=(proc, stop, end), daemon=True)
        t_start = el()
        th.start()
        wd.start()
        with ThreadPoolExecutor(max_workers=conc) as pool:
            list(pool.map(lambda j: one_request(j, 8001, end, stop_at=end), xs[:conc]))
        stop.set()
        th.join(timeout=30)
        wd.join(timeout=30)
        rows = [json.loads(x) for x in (OUT / "metrics.jsonl").read_text().splitlines() if '"tp2"' in x]
        if len(rows) >= 2:
            dt = max(1.0, rows[-1]["t"] - rows[0]["t"])
            ph["short_context_tok_per_s"] = round((rows[-1].get("vllm:generation_tokens_total", 0)
                                                   - rows[0].get("vllm:generation_tokens_total", 0)) / dt, 1)
        ph.update(concurrency=conc, t_start=t_start)
    log(f"tp2 tail: {json.dumps(ph)}")
    save()
    kill(proc)


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    proc = None
    try:
        comp, model, wheels = phase_env()
        ref, futs, pool = start_model_hash(model)
        HASH_POOL.append(pool)
        verify_wheels(wheels)
        install(wheels)
        finish_model_hash(model, ref, futs, pool)
        proc, att = phase_server(model)
        phase_main(comp, att, proc=proc)
        kill(proc)
        proc = None
        if T0 + WALL - time.time() > TAIL_NEED:
            phase_tp2(model, comp, att)
        else:
            S["phases"]["tp2"] = {"skipped": "not enough time"}
    except SystemExit as e:
        S["fatal"] = str(e)
        log(f"FATAL: {e}")
    except Exception as e:  # noqa: BLE001
        import traceback
        S["fatal"] = repr(e)
        S["traceback"] = traceback.format_exc()[-4000:]
        log(f"EXCEPTION: {e!r}")
    finally:
        ABORT.set()
        for pool_ in HASH_POOL:
            pool_.shutdown(wait=False, cancel_futures=True)
        kill(proc)
        S["elapsed_seconds"] = el()
        save()
        log("probe finished")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile keys30.json
["dbff022c", "3dc255db", "2b83f449", "5545f144", "dd6b8c4b", "f931b4a8", "53fb4810", "332f06d7", "d35bdbdc", "58f5dbd5", "faa9f03d", "dfadab01", "80a900e0", "581f7754", "446ef5d2", "7b0280bc", "a251c730", "2c181942", "4e34c42c", "7ed72f31", "b6f77b65", "38007db0", "4a21e3da", "142ca369", "cb2d8a2c", "6e4f6532", "13e47133", "e12f9a14", "3e6067c3", "e3721c99"]

In [ ]:
%%writefile qwen38_fp8_sha256.json
{"source":"https://huggingface.co/api/models/Qwen/Qwen3.8-27B-FP8/tree/017b9c7af6b5689d5dd426a76e0bc077eb5ca20a","fetched_at":"2026-10-12T08:05Z","revision":"017b9c7af6b5689d5dd426a76e0bc077eb5ca20a","kaggle_mirror":"mikedan7/qwen3-8-27b-fp8-official/PyTorch/hf-fp8/1 (same file names and sizes; README says unmodified mirror at revision 017b9c7af6b5)","files":{"layers-0.safetensors":[383865448,"07f700e293baeaf3cd4240c3df1a948c4403f16961ea7979e86c8d6a9f8fd466"],"layers-1.safetensors":[383865448,"35840b5d452c6e438d000e5c1d8d1bc793d257394404689b8f9424749eee8edc"],"layers-10.safetensors":[383865472,"ab5fbc076ccd6514aba3ab67a0ac1d6701ad395d40cccc0086cf0284ef5d68c9"],"layers-11.safetensors":[372313760,"076b4ba44e9311c8fb8fdc0045f8ce9fd0c75584ade4e95510056b7db4b7ec61"],"layers-12.safetensors":[383865472,"86d8ec0b8fc8e3ffbdfd667b97050533ddf5a4d273b4fe1190a4335542c52829"],"layers-13.safetensors":[383865472,"71dd986134f4338a7b8290ed24993ad34a0f77b76a4cc060fe6403e36dcdda5b"],"layers-14.safetensors":[383865472,"95b285915cba994917b527030874227382ea3421dc7f47d804c0ca5851f039fb"],"layers-15.safetensors":[372313760,"1e87d61e77d2b802f796ab8a049b1119648c9b70d2ba1067f456e5a04449cfc3"],"layers-16.safetensors":[383865472,"8bcdbf4e2c7a8dda3043c4453a476b30a582d58aabfaa49b2829a8b66a664acb"],"layers-17.safetensors":[383865472,"571cea44717878cc8cbbfbd544acfe2d155f5b963fbf5dc67b3fa96afe840a1b"],"layers-18.safetensors":[383865472,"d8700627ca2267fb7beb5d398d6a659a9bfeeb2af76f10e6827b1e4359d59286"],"layers-19.safetensors":[372313760,"2b24cfb752bbed959c614d7fd85412ee27ab2ac1f47deef5d09d0592781400e4"],"layers-2.safetensors":[383865448,"32e63c6455ebfa10b4c36af2878eefcb6f8ccf8e947a62575444dc412c38ac1a"],"layers-20.safetensors":[383865472,"52fd00c68d4f1df96ba701290c04237257ebb832879bb0b6a629525809d84a64"],"layers-21.safetensors":[383865472,"e3c29dd949808c13116d853ce4b0f96cb1238fc0cb6999ab76735deef6bda77a"],"layers-22.safetensors":[383865472,"63446546e3e09c2394fc51855f4e390cb47f896bb8b744f3a210c3159a3ccaf2"],"layers-23.safetensors":[372313760,"4166ed11d0329985f9813ba6a70474ea9f5aeff3d97db3d7b2e81185bd41a996"],"layers-24.safetensors":[383865472,"720bdcfe12b6627bed29feda9c96256e15c20602704669d05097668a9f03dd45"],"layers-25.safetensors":[383865472,"1b188e7e4e8ae6f5d753c597f61d86c1593495de4307c273d8dda9946ec48bb4"],"layers-26.safetensors":[383865472,"ad8c3e3d4b79dfedad674318ee4b150b2cd78b56747c7c8d88a65818109a545d"],"layers-27.safetensors":[372313760,"8d63f461430045960f27a22abf685bfc7445fffbb24764e4a7fa993ba7edac08"],"layers-28.safetensors":[383865472,"f78557cda66107ce25f8c5ecde2f83da874b3bf7c50381e062e14e7294a5b64d"],"layers-29.safetensors":[383865472,"09f264d6fdddab0fb5ecce18034a9a01944d15118147c97058fba101a60f8af1"],"layers-3.safetensors":[372313744,"302f9af90bb683a8be9e96d124b470a2eddee6612c95c39a8e93f26eb654563d"],"layers-30.safetensors":[383865472,"54afe4fc7262ee597d4d9b80d2d6fe8ae501846ff124334b575bed4c6850ed97"],"layers-31.safetensors":[372313760,"2779338203d0715ca323ac0b7fdbb0d450ce2bd1694be395a211a02ade3f34d9"],"layers-32.safetensors":[383865472,"f4af8893594458b6cdb074ef20f9accb708361fe904012465990d09e74994509"],"layers-33.safetensors":[383865472,"8f038505b9fc7d0b31ca976c728b56028ec28390a15ca0410f00df18d3abc1a4"],"layers-34.safetensors":[383865472,"0b28f43cc4d3ae50e37eb3c2ce12a6e181b1c7586a27d5a05c6a3367f08286c9"],"layers-35.safetensors":[372313760,"535b61b9eabbcda5de20e21d2d46fb52528cc9c1f8fac38920382ea2f1030837"],"layers-36.safetensors":[383865472,"2babd209cafbe12a6e375ff6e7a1029f36ee8226f2a9babf6646630f31dd3fb8"],"layers-37.safetensors":[383865472,"651cae579808983fefec954db6e2ac8c3f98bfb038ff1d50aef0505ba7394e65"],"layers-38.safetensors":[383865472,"1c60932dec650cc7679121eaa4d90b8d10a600bf58e23b0784588a339469dd0a"],"layers-39.safetensors":[372313760,"b0c7df4d51b99637ff4ea35eebf7eb134db592eb943e3ccde921729f33949058"],"layers-4.safetensors":[383865448,"b7f367125cdb4b3c3920d1d3ed1ba0d73e464c3e3a92dbd1c195658cb9200afd"],"layers-40.safetensors":[383865472,"6908f1ab1a3d7828a566099ecf393582bec988d0f82f6141848942d1494b0321"],"layers-41.safetensors":[383865472,"21ee5d9842074888bd9fdffa875d1f29f8c77f1f5ccbb8f53becb188e1749caa"],"layers-42.safetensors":[383865472,"565379d4291c06cdf9a4f66d5d21385848a1e5ab17b55fea6c1c8e544f8498f0"],"layers-43.safetensors":[372313760,"4bb8c0a4fadda1f1d66ef6f0dde5a5a94001d04af1842e42009833aff70bf416"],"layers-44.safetensors":[383865472,"14339304cd520a8673f179cb8352c36fbebe184f15309aee2a79afd2d4476fef"],"layers-45.safetensors":[383865472,"7a076aca542f937cc87dec92ab49fd782eae234642b40bc706a6108b3dff2c2a"],"layers-46.safetensors":[383865472,"d82d472cfc4792012934467b90c5e0242df4867a430dd651d13cdee72b7c6645"],"layers-47.safetensors":[372313760,"a83fc757fb11bc50ed3a0c9998b2b2f55c3b6a07598ed458bdbece4285ffaf1f"],"layers-48.safetensors":[383865472,"f6aa126d006c7d976c20b1330c118b42cf3bbe8a7d216bb14f2240d5c49bc72f"],"layers-49.safetensors":[383865472,"7ed54c375407c2baec8cfabe8336f8d3782246848298bb232ac2582113676c69"],"layers-5.safetensors":[383865448,"ebbc2c4bd2b98877caafa7073b6ed732a0284eb82269cf6d21710e245af5837a"],"layers-50.safetensors":[383865472,"73825e1f056c3de53276d3c65743e62b67bf3ca082cf7077ecba1ffef18e9bf1"],"layers-51.safetensors":[372313760,"b6abd8b62b5c4b7beca8cdccd306065be2192ef42491c587ca019d971ed86a1f"],"layers-52.safetensors":[383865472,"6251d51e6fbec6f463ca875fcbe852e07c5025145affe9af053a04fbdc37288f"],"layers-53.safetensors":[383865472,"a9f2f43b4978bba2f755c6d2eb39f8561965bc3c1b2fd1073bb363aff11a9b8c"],"layers-54.safetensors":[383865472,"f0246c9974a4b1207f8f5d05a09c54c50ea61fbfd2dc6480d2b76d3647ece292"],"layers-55.safetensors":[372313760,"649c400495616cf888b7c52e538113610a36265c531d84bff484f8149c157405"],"layers-56.safetensors":[383865472,"eb0a8139a36138639a37219c158b9415412b68d316235061a72e57da5509c188"],"layers-57.safetensors":[383865472,"d7616ad33f9256342573ff85b3f2a0e368dd120257c0c7158564587036119d95"],"layers-58.safetensors":[383865472,"a8dff0e37d0e3903cc29711b935c2865ccd38bd930e7487ae2f4bedbd18ff185"],"layers-59.safetensors":[372313760,"573ed65fd455ec9bd811e95946ea422e08ac36860fdff6968a6d108263a54175"],"layers-6.safetensors":[383865448,"2fea2aaa61d566ba8af27ee89d29d9ee9f1c7f6fbe692772d9ba61fc45109ef6"],"layers-60.safetensors":[383865472,"9a19c30b190bf552eb424af16103f610bcd944fb276b3275eb7c73849799b7de"],"layers-61.safetensors":[383865472,"ff744debd4dd0eef450ed24473df57680db404914302e7147a3b72154131f4a0"],"layers-62.safetensors":[383865472,"d9b3b9e472f78bcadf62c446084833bb549faff9e730b21219f1f80d583eff23"],"layers-63.safetensors":[372313760,"59ba4a3af5e6bc2008c2d6b5d9be8eeaf255cddc0aaf8a0d9583b496bb2e3ae0"],"layers-7.safetensors":[372313744,"f2e0137e878016a7afdb2314972e0cbbe2461a67b00eb1d691b4de8ea2665da8"],"layers-8.safetensors":[383865448,"1721b7bcd730891c224b70a7b6147e44a574a2d6c4d924760acf97728249c92c"],"layers-9.safetensors":[383865448,"0a21e07065b5bb04adc339300dea1872cc48ab8eab3b02df092f7cbc67c8d6f6"],"mtp.safetensors":[477202224,"e5e4464a3793cc261de536592830bca40e7f3af159ed038c358f5660917cf43b"],"outside.safetensors":[6007102112,"ddff1d6665a2b39f2612fce0ef955e2436724c565bfbcbc127c7ffd078b698ff"],"tokenizer.json":[12809320,"0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3"]}}


In [ ]:
!cd /kaggle/working && python p0_probe.py
